In [1]:
import os
import sys
import logging
from datetime import datetime  
import pandas as pd
from configs import MakeIdsMapTableConfig
from make_ids_map_table_utils import *

In [2]:
if not os.path.exists(MakeIdsMapTableConfig.RAW_DATASET):
    logging.error(f"文件未找到: {MakeIdsMapTableConfig.RAW_DATASET}")
    sys.exit(1)

logging.info("开始加载数据集...")
file_ext = os.path.splitext(MakeIdsMapTableConfig.RAW_DATASET)[1].lower()
try:
    if file_ext == '.csv':
        raw_dataset = pd.read_csv(MakeIdsMapTableConfig.RAW_DATASET, encoding='utf-8-sig') 
    elif file_ext in ['.xlsx', '.xls']:
        raw_dataset = pd.read_excel(MakeIdsMapTableConfig.RAW_DATASET)
except Exception as e:
    print(f"读取文件出错: {e}")
    sys.exit(1)

compound_array = pd.unique(raw_dataset[['c1', 'c2', 'c3']].values.ravel())
compounds_table = pd.DataFrame(compound_array, columns=['compound']).dropna()
compounds_table = compounds_table[compounds_table['compound'].astype(str).str.strip() != '']

total_compounds = len(compounds_table)
logging.info(f"提取到 {total_compounds} 个需要处理的唯一化合物。")

results_dict = {} # 使用字典存储结果，方便后续根据 name 更新

2026-11-18 14:58:29,304 - INFO - 开始加载数据集...
2026-11-18 14:58:29,513 - INFO - 提取到 445 个需要处理的唯一化合物。


In [3]:
# ---------------- Phase 1: 初次全量检索 ----------------
logging.info("=== 启动第一轮全量检索 ===")
for idx, name in enumerate(compounds_table['compound'], start=1):
    print(f"\n" + "="*60)
    print(f"初次检索进度: [{idx}/{total_compounds}] | 当前处理: {name}")
    
    percent = idx / total_compounds
    filled_length = int(40 * percent)
    bar = '█' * filled_length + '-' * (40 - filled_length)
    print(f"总体进度: |{bar}| {percent*100:.1f}%")
    print(f"--> 正在向 PubChem 请求 ...", flush=True)
    
    res = f_fetch_compound_info(name)
    results_dict[name] = res

2026-11-18 14:58:32,461 - INFO - === 启动第一轮全量检索 ===

初次检索进度: [1/445] | 当前处理: Choline chloride
总体进度: |----------------------------------------| 0.2%
--> 正在向 PubChem 请求 ...
2026-11-18 14:58:32,963 - INFO - --> 正在请求: [Choline chloride] ...


d:\envs\codes\DES-DataAlchemy\dataset_pretreatment\src\make_ids_map_table_utils.py:101: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  data['ismile'] = compound.isomeric_smiles


2026-11-18 14:58:35,179 - INFO -     └─ 成功获取 [ismile]: C[N+](C)(C)CCO.[Cl-]
2026-11-18 14:58:35,180 - INFO -     └─ 成功获取 [uname]: 2-hydroxyethyl(trimethyl)azanium chloride
2026-11-18 14:58:35,180 - INFO -     └─ 成功获取 [cas]: 67-48-1
2026-11-18 14:58:35,181 - INFO - Result 'Choline chloride': ismile=C[N+](C)(C)CCO.[Cl-], uname=2-hydroxyethyl(trimethyl)azanium chloride, CAS=67-48-1

初次检索进度: [2/445] | 当前处理: Glycerol
总体进度: |----------------------------------------| 0.4%
--> 正在向 PubChem 请求 ...
2026-11-18 14:58:35,682 - INFO - --> 正在请求: [Glycerol] ...
2026-11-18 14:58:39,128 - INFO -     └─ 成功获取 [ismile]: C(C(CO)O)O
2026-11-18 14:58:39,130 - INFO -     └─ 成功获取 [uname]: propane-1,2,3-triol
2026-11-18 14:58:39,132 - INFO -     └─ 成功获取 [cas]: 56-81-5
2026-11-18 14:58:39,133 - INFO - Result 'Glycerol': ismile=C(C(CO)O)O, uname=propane-1,2,3-triol, CAS=56-81-5

初次检索进度: [3/445] | 当前处理: Ethylene glycol
总体进度: |----------------------------------------| 0.7%
--> 正在向 PubChem 请求 ...
2026-11-18 14:58:39,6

In [4]:
# ---------------- Phase 2: 审查与二次回炉 ----------------
failed_names = []
for name, res in results_dict.items():
    # 如果 smiles, uname, cas 中有任何一个是 None，就判定为失败/不完整
    if None in res:
        failed_names.append(name)

if failed_names:
    logging.warning(f"\n⚠️ 第一轮检索结束。发现 {len(failed_names)} 个化合物的数据不完整！准备打回重审...")
    # 给服务器一点喘息的时间，防止是因为被封锁导致的集体失败
    cooldown_time = 10 
    logging.info(f"冷静期：休眠 {cooldown_time} 秒后启动第二轮检索...")
    time.sleep(cooldown_time)
    
    # 清除缓存，强制让这些失败的化合物重新去网络上查一遍，而不是读取刚才失败的记录
    f_fetch_compound_info.cache_clear()
    
    logging.info("=== 启动第二轮补漏检索 (Retry-on-Miss) ===")
    total_retry = len(failed_names)
    for idx, name in enumerate(failed_names, start=1):
        print(f"\n" + "="*60)
        print(f"🚑 补漏进度: [{idx}/{total_retry}] | 正在重新抢救: {name}")
        
        # 在重试前额外增加一个休眠，稳扎稳打
        time.sleep(2) 
        print(f"--> 正在再次向 PubChem 发起请求 ...", flush=True)
        
        retry_res = f_fetch_compound_info(name)
        
        # 覆盖刚才失败的记录
        results_dict[name] = retry_res
        
        if None not in retry_res:
            logging.info(f"🎉 抢救成功！已成功补齐 {name} 的所有字段。")
        else:
            logging.warning(f"❌ 抢救失败，{name} 的数据在 PubChem 中可能确实不存在或存在持续网络阻断。")
else:
    logging.info("\n🎉 第一轮检索非常完美！所有化合物数据均完整，无需重试。")

2026-11-18 15:22:13,000 - WARNING - 
⚠️ 第一轮检索结束。发现 67 个化合物的数据不完整！准备打回重审...
2026-11-18 15:22:13,000 - INFO - 冷静期：休眠 10 秒后启动第二轮检索...
2026-11-18 15:22:23,001 - INFO - === 启动第二轮补漏检索 (Retry-on-Miss) ===

🚑 补漏进度: [1/67] | 正在重新抢救: allyl triethyl ammonium chloride
--> 正在再次向 PubChem 发起请求 ...
2026-11-18 15:22:25,504 - INFO - --> 正在请求: [allyl triethyl ammonium chloride] ...
2026-11-18 15:22:26,410 - INFO - PubChem HTTP Error 404 PUGREST.NotFound: No CID found (No CID found that matches the given name)
2026-11-18 15:22:26,411 - INFO - Result 'allyl triethyl ammonium chloride': ismile=None, uname=None, CAS=None
2026-11-18 15:22:26,412 - WARNING - ❌ 抢救失败，allyl triethyl ammonium chloride 的数据在 PubChem 中可能确实不存在或存在持续网络阻断。

🚑 补漏进度: [2/67] | 正在重新抢救: N,N,N,N-tetraoctylammonium bromide
--> 正在再次向 PubChem 发起请求 ...
2026-11-18 15:22:28,914 - INFO - --> 正在请求: [N,N,N,N-tetraoctylammonium bromide] ...
2026-11-18 15:22:29,840 - INFO - PubChem HTTP Error 404 PUGREST.NotFound: No CID found (No CID found that matches 

In [5]:
# ---------------- Phase 3: 整理与保存 ----------------
print(f"\n" + "="*60)
logging.info("所有处理完毕，正在整合并保存数据...")

# 按照原本 DataFrame 的顺序把结果抽出来
final_results = [results_dict[name] for name in compounds_table['compound']]

result_df = pd.DataFrame(
    final_results, 
    index=compounds_table.index, 
    columns=['ismiles', 'uname', 'cas']
)

ids_map_table = pd.concat([compounds_table, result_df], axis=1)

qlty_sum_output_path = os.path.join(MakeIdsMapTableConfig.MISSING_DATA_MAG_DIR, 'ids-map-tb_qlty-sum.xlsx')
data_quality_summary(ids_map_table, qlty_sum_output_path)

os.makedirs(os.path.dirname(MakeIdsMapTableConfig.IDS_MAP_TABLE) or '.', exist_ok=True)
ids_map_table.to_excel(MakeIdsMapTableConfig.IDS_MAP_TABLE, index=False)

logging.info(f"✅ 处理完成！分子名称映射表已保存至: {MakeIdsMapTableConfig.IDS_MAP_TABLE}")

print("如有需要请手动填补空缺的标识符")


2026-11-18 15:26:55,450 - INFO - 所有处理完毕，正在整合并保存数据...
2026-11-18 15:26:56,339 - INFO - ✅ 处理完成！分子名称映射表已保存至: D:\envs\codes\DES-DataAlchemy\dataset_pretreatment\data\in_out\alpha_dataset_26-09-11_261118_ids-map-table.xlsx
如有需要请手动填补空缺的标识符


In [6]:
group_cols = ['ismiles', 'uname', 'cas']
byproduct_table = deduplicate_byproduct_table(ids_map_table, group_cols, MakeIdsMapTableConfig.DEDUPLICATE_DIR )
byproduct_table.to_excel(MakeIdsMapTableConfig.BYPRODUCT_TABLE , index=False)

涉及重复的总行数: 287
重复明细已整合并保存至: D:\envs\codes\DES-DataAlchemy\dataset_pretreatment\tmp_data\alpha_dataset_26-09-11\deduplicate\byproduct_duplicated_details.xlsx (单元格已合并)
开始去重，规则：直接留一
原始行数: 445
去重后行数: 240
